In [ ]:
import os
os.environ["GROQ_API_KEY"]="....."

In [ ]:
!pip install langchain_groq  dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


In [ ]:
#tool creaete
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def muiltply(a: int,b:int):
  """Given 2 numbers a and b this tool return their product """
  return a*b

In [ ]:
print(muiltply.invoke({'a':2,'b':3}))

6


In [ ]:
muiltply.name

'muiltply'

In [ ]:
muiltply.description


'Given 2 numbers a and b this tool return their product'

In [ ]:
muiltply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [ ]:
#tool binding

In [ ]:

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.9,
)

In [ ]:
llm_with_tools = llm.bind_tools([muiltply])

In [ ]:
query=HumanMessage("can you muiltply 2 with 10")

In [ ]:
messages=[query]

In [ ]:
messages

[HumanMessage(content='can you muiltply 2 with 10', additional_kwargs={}, response_metadata={})]

In [ ]:
result = llm_with_tools.invoke(messages)

In [ ]:
messages.append(result)

In [ ]:
messages

[HumanMessage(content='can you muiltply 2 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to multiply 2 with 10. Let me check the available tools. There\'s a function called \'muiltply\' which takes two integers a and b. The parameters required are a and b, both integers. The user provided the numbers 2 and 10. I need to call the \'muiltply\' function with a=2 and b=10. Let me make sure the parameters are correct. Yes, they are integers. So the tool call should be {"name": "muiltply", "arguments": {"a": 2, "b": 10}}.\n', 'tool_calls': [{'id': 'jwbahrs3g', 'function': {'arguments': '{"a":2,"b":10}', 'name': 'muiltply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 178, 'total_tokens': 344, 'completion_time': 0.253804622, 'completion_tokens_details': {'reasoning_tokens': 133}, 'prompt_time': 0.007414166, 'prompt_tokens_details': None, 'qu

In [ ]:
tool_result = muiltply.invoke(result.tool_calls[0])

In [ ]:
messages.append(tool_result)

In [ ]:
messages

[HumanMessage(content='can you muiltply 2 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to multiply 2 with 10. Let me check the available tools. There\'s a function called \'muiltply\' which takes two integers a and b. The parameters required are a and b, both integers. The user provided the numbers 2 and 10. I need to call the \'muiltply\' function with a=2 and b=10. Let me make sure the parameters are correct. Yes, they are integers. So the tool call should be {"name": "muiltply", "arguments": {"a": 2, "b": 10}}.\n', 'tool_calls': [{'id': 'jwbahrs3g', 'function': {'arguments': '{"a":2,"b":10}', 'name': 'muiltply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 178, 'total_tokens': 344, 'completion_time': 0.253804622, 'completion_tokens_details': {'reasoning_tokens': 133}, 'prompt_time': 0.007414166, 'prompt_tokens_details': None, 'qu

In [ ]:
llm_with_tools.invoke(messages).content

'The product of 2 and 10 is **20**.'

In [ ]:
import requests
from langchain.tools import tool

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Fetch the currency conversion factor between the given base currency
    and target currency.
    """
    url = f"https://v6.exchangerate-api.com/v6/066fd6d763bd1251f17aca26/latest/{base_currency}"

    response = requests.get(url)
    data = response.json()

    if data["result"] == "success":
        return data["conversion_rates"][target_currency]
    else:
        raise Exception(data.get("error-type", "Failed to fetch exchange rate"))

def convert(base_curreny_value: int,conversion_rate: Annotated[float,InjectedToolArg])-> float:
  """
  given a curreny conversion rate this function calculates the target currency from  a gven base curreny value
  """
  return base_curreny_value*conversion_rate




In [ ]:
rate = get_conversion_factor.invoke({
    "base_currency": "USD",
    "target_currency": "INR"
})

converted_amount = convert(10, rate)

print(converted_amount)

957.212


In [ ]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.9,
)

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [ ]:
messages=[HumanMessage("can you convert 10 USD to INR")]

In [ ]:
messages

[HumanMessage(content='can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [ ]:
ai_message= llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'h83k257z6',
  'type': 'tool_call'}]

In [ ]:
import json
for tool_call in ai_message.tool_calls:
  #execute the 1 st tool and get the value of conversion rate4
  if tool_call['name'] == "get_conversion_factor":
      tool_message1 = get_conversion_factor.invoke(tool_call)
      #fetch this conversion rate
      conversion_rate = json.loads(tool_message1.content)
      # print(conversion_rate)
      #append this tool message to message list
      messages.append(tool_message1)
  #execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == "convert":
      #fetch this conversion rate
      tool_call['args']['conversion_rate']= conversion_rate
      tool_message2 = convert.invoke(tool_call)
      messages.append(tool_message2)
      # print(tool_message2)
      #append this tool message to message list


In [ ]:
messages

[HumanMessage(content='can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='95.7212', name='get_conversion_factor', tool_call_id='h83k257z6'),
 ToolMessage(content='95.7212', name='get_conversion_factor', tool_call_id='h83k257z6'),
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to convert 10 USD to INR. Let me check the available functions. There's get_conversion_factor which requires base and target currencies. Then there's convert which needs the base currency value. First, I need to get the conversion rate between USD and INR using get_conversion_factor. Once I have that rate, I can use the convert function to calculate the INR value for 10 USD. So I'll start by calling get_conversion_factor with base_currency as USD and target_currency as INR.\n", 'tool_calls': [{'id': 'h83k257z6', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 

In [ ]:
llm_with_tools.invoke(messages).content

'The conversion of 10 USD to INR is **957.21 INR** (using the conversion rate of 1 USD = 95.7212 INR).'